# CNN From Scratch vs Keras

## Import

In [1]:
import os
import sys
import glob
import time
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score

sys.path.append('../../../')

from src.cnn.utils import load_image, batch_load_images
from src.cnn.model import CNNModel

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TF version:', tf.__version__)

I0000 00:00:1778833824.481102    3053 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.21.0


## Config

In [2]:
TEST_DIR   = '../../data/intel/seg_test'
MODEL_PATH = 'Layer-2-32-64-5x5-maxpool.h5'

IMG_SIZE    = (150, 150)
NUM_CLASSES = 6
CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

## Load Test

In [3]:
all_paths, all_labels = [], []
for label_idx, cls in enumerate(CLASS_NAMES):
    paths = sorted(glob.glob(os.path.join(TEST_DIR, cls, '*.jpg')))
    all_paths.extend(paths)
    all_labels.extend([label_idx] * len(paths))

X_test = batch_load_images(all_paths, target_size=IMG_SIZE)
y_true = np.array(all_labels, dtype=int)

print(f'X_test : {X_test.shape}')
print(f'y_true : {y_true.shape}')

X_test : (3000, 150, 150, 3)
y_true : (3000,)


## Load Keras Model

In [4]:
keras_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
keras_model.summary()

I0000 00:00:1778834003.928386    3053 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "Layer-2-32-64-5x5-maxpool"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2s_1 (Conv2D)               │ (None, 146, 146, 32)   │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling2D)        │ (None, 73, 73, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2s_2 (Conv2D)               │ (None, 69, 69, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling2D)        │ (None, 34, 34, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 73984)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │     9,470,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,524,550 (36.33 MB)

 Trainable params: 9,524,550 (36.33 MB)

 Non-trainable params: 0 (0.00 B)

## Evaluasi Keras

In [5]:
t0 = time.perf_counter()
y_pred_keras = np.argmax(
    keras_model.predict(X_test, batch_size=32, verbose=1),
    axis=1
)
t_keras = time.perf_counter() - t0

f1_keras = f1_score(y_true, y_pred_keras, average='macro')
print(f'\nKeras Macro F1 : {f1_keras:.4f}')
print(f'Keras Time     : {t_keras:.2f}s')

W0000 00:00:1778834011.361358    3053 cpu_allocator_impl.cc:82] Allocation of 810000000 exceeds 10% of free system memory.
W0000 00:00:1778834013.797318    3053 cpu_allocator_impl.cc:82] Allocation of 810000000 exceeds 10% of free system memory.
I0000 00:00:1778834015.388787    3876 service.cc:153] XLA service 0x7d4ccc002c90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778834015.388863    3876 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.5.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1778834015.501562    3876 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778834015.681645    3876 cuda_dnn.cc:461] Loaded cuDNN version 92200


 5/94 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step

I0000 00:00:1778834021.922292    3876 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


94/94 ━━━━━━━━━━━━━━━━━━━━ 11s 46ms/step

Keras Macro F1 : 0.7839
Keras Time     : 16.38s


## Build Scratch Model (transfer bobot)

In [6]:
scratch_model = CNNModel(keras_model)

# Sanity check
probe = scratch_model.forward(X_test[0])
assert probe.shape == (NUM_CLASSES,)
assert abs(probe.sum() - 1.0) < 1e-4
print('Sanity check OK probe shape:', probe.shape, '| sum:', round(float(probe.sum()), 6))

Sanity check OK probe shape: (6,) | sum: 1.0


## Evaluasi Scratch

In [7]:
n = len(X_test)
y_pred_scratch = np.zeros(n, dtype=int)
t0 = time.perf_counter()

for i, img in enumerate(X_test):
    y_pred_scratch[i] = scratch_model.predict(img)
    if (i + 1) % 500 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (i + 1) * (n - i - 1)
        print(f'  {i+1}/{n} | {elapsed:.1f}s elapsed | ETA: {eta:.1f}s')

t_scratch = time.perf_counter() - t0
f1_scratch = f1_score(y_true, y_pred_scratch, average='macro')

print(f'\nScratch Macro F1 : {f1_scratch:.4f}')
print(f'Scratch Time     : {t_scratch:.2f}s')

  500/3000 | 334.8s elapsed | ETA: 1674.2s
  1000/3000 | 651.8s elapsed | ETA: 1303.6s
  1500/3000 | 961.9s elapsed | ETA: 961.9s
  2000/3000 | 1269.1s elapsed | ETA: 634.6s
  2500/3000 | 1584.4s elapsed | ETA: 316.9s
  3000/3000 | 1900.9s elapsed | ETA: 0.0s

Scratch Macro F1 : 0.7838
Scratch Time     : 1900.95s


## Hasil Perbandingan

In [8]:
print(f'>>> Keras Macro F1   : {f1_keras:.4f}')
print(f'>>> Scratch Macro F1 : {f1_scratch:.4f}')
print(f'>>> Selisih          : {abs(f1_keras - f1_scratch):.6f}')

>>> Keras Macro F1   : 0.7839
>>> Scratch Macro F1 : 0.7838
>>> Selisih          : 0.000056
